In [1]:
import pandas as pd

def compute_offer_funnel_metrics(df: pd.DataFrame) -> dict:
    """
    Computes key conversion rates and drop-off metrics for Task 11 Offer & E-Sign Workflow.
    """
    total_created = len(df)
    total_delivered = len(df[df['delivered_at'].notnull()])
    total_viewed = len(df[df['viewed_at'].notnull()])
    total_accepted = len(df[df['status'] == 'ACCEPTED'])
    total_signed = len(df[df['signed_at'].notnull()])
    total_expired = len(df[df['status'] == 'EXPIRED'])
    
    return {
        "Total Offers Created": total_created,
        "Delivery Rate": f"{(total_delivered / total_created) * 100:.1f}%",
        "View Rate": f"{(total_viewed / total_delivered) * 100:.1f}%",
        "Acceptance Rate": f"{(total_accepted / total_viewed) * 100:.1f}%",
        "E-Sign Completion Rate": f"{(total_signed / total_accepted) * 100:.1f}%",
        "Overall Funnel Yield": f"{(total_signed / total_created) * 100:.1f}%",
        "Expiry Drop-off Rate": f"{(total_expired / total_delivered) * 100:.1f}%"
    }

In [2]:
import numpy as np

# 1. Generate Mock Offer Dataset
np.random.seed(42)
n_offers = 100

dates = pd.date_range(start="2026-08-01", periods=n_offers, freq="h")
data = {
    'offer_id': [f"OFF-{1000+i}" for i in range(n_offers)],
    'created_at': dates,
    'delivered_at': [d + pd.Timedelta(minutes=5) if np.random.rand() > 0.05 else None for d in dates],
    'viewed_at': [d + pd.Timedelta(hours=2) if np.random.rand() > 0.15 else None for d in dates],
    'status': np.random.choice(['ACCEPTED', 'REJECTED', 'EXPIRED'], size=n_offers, p=[0.65, 0.20, 0.15]),
}

df_offers = pd.DataFrame(data)
df_offers['signed_at'] = [
    row['viewed_at'] + pd.Timedelta(hours=12) 
    if row['status'] == 'ACCEPTED' and row['viewed_at'] is not None 
    else None 
    for _, row in df_offers.iterrows()
]

# 2. Execute Funnel Calculation Function
metrics_summary = compute_offer_funnel_metrics(df_offers)

# 3. Format & Render Results
results_df = pd.DataFrame(list(metrics_summary.items()), columns=['Funnel Metric', 'Value'])
results_df

,Funnel Metric,Value
0,Total Offers Created,100
1,Delivery Rate,94.0%
2,View Rate,91.5%
3,Acceptance Rate,73.3%
4,E-Sign Completion Rate,88.9%
5,Overall Funnel Yield,56.0%
6,Expiry Drop-off Rate,18.1%
